<a href="https://colab.research.google.com/github/SJH-official/Application_Developments/blob/main/load_wine()_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EC%98%88%EC%B8%A1_%EC%84%9C%EB%B9%84%EC%8A%A4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install gradio scikit-learn pandas numpy matplotlib -q

In [3]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import io
from PIL import Image
import gradio as gr

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

In [4]:
iris = load_iris()
X_raw = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target
class_names = list(iris.target_names)

print(f'샘플 수: {len(y)}개')
print(f'클래스: {class_names}')
X_raw.head()

샘플 수: 150개
클래스: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [6]:
# Step 1: 표준화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Step 2: Feature Selection - 상위 3개 선택
selector = SelectKBest(score_func=f_classif, k=3)
X_selected = selector.fit_transform(X_scaled, y)

selected_mask     = selector.get_support()
feature_scores    = selector.scores_
selected_features = [f for f, m in zip(iris.feature_names, selected_mask) if m]
removed_features  = [f for f, m in zip(iris.feature_names, selected_mask) if not m]

score_df = pd.DataFrame({
    '피처명': iris.feature_names,
    'F-score': feature_scores.round(2),
    '선택 여부': ['✅ 선택' if m else '❌ 제거' for m in selected_mask]
}).sort_values('F-score', ascending=False)
print(score_df.to_string(index=False))
print(f'\n선택된 피처: {selected_features}')
print(f'제거된 피처: {removed_features}')

              피처명  F-score 선택 여부
petal length (cm)  1180.16  ✅ 선택
 petal width (cm)   960.01  ✅ 선택
sepal length (cm)   119.26  ✅ 선택
 sepal width (cm)    49.16  ❌ 제거

선택된 피처: ['sepal length (cm)', 'petal length (cm)', 'petal width (cm)']
제거된 피처: ['sepal width (cm)']


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred   = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
report   = classification_report(y_test, y_pred, target_names=class_names)

print(f'테스트 정확도: {accuracy*100:.2f}%')
print(report)

테스트 정확도: 90.00%
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.82      0.90      0.86        10
   virginica       0.89      0.80      0.84        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30



In [8]:
PALETTE = ['#4C72B0', '#DD8452', '#55A868']
BG = '#F7F8FA'

def fig_to_pil(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=130, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    buf.seek(0)
    img = Image.open(buf).copy()
    buf.close()
    plt.close(fig)
    return img

def plot_feature_scores():
    fig, ax = plt.subplots(figsize=(6, 3.6), facecolor=BG)
    colors = [PALETTE[0] if m else '#CCCCCC' for m in selected_mask]
    bars = ax.bar(iris.feature_names, feature_scores, color=colors,
                  edgecolor='white', width=0.5)
    for bar, val, m in zip(bars, feature_scores, selected_mask):
        label = f'{val:.1f}{" ✓" if m else ""}'
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
                label, ha='center', va='bottom', fontsize=9,
                color=PALETTE[0] if m else '#999')
    ax.set_ylabel('F-score', fontsize=11)
    ax.set_title('Feature Selection 점수 (파란색 = 선택됨)', fontsize=12, fontweight='bold')
    ax.set_xticklabels(iris.feature_names, rotation=15, ha='right', fontsize=9)
    ax.set_facecolor(BG)
    ax.spines[['top','right']].set_visible(False)
    fig.tight_layout()
    return fig_to_pil(fig)

def plot_feature_importance():
    importances = model.feature_importances_
    fig, ax = plt.subplots(figsize=(6, 3.2), facecolor=BG)
    bars = ax.barh(selected_features, importances,
                   color=PALETTE, edgecolor='white', height=0.45)
    for bar, val in zip(bars, importances):
        ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                f'{val:.3f}', va='center', fontsize=10)
    ax.set_xlabel('Importance', fontsize=11)
    ax.set_title('피처 중요도 (Random Forest)', fontsize=12, fontweight='bold')
    ax.set_facecolor(BG)
    ax.spines[['top','right']].set_visible(False)
    ax.set_xlim(0, max(importances) * 1.25)
    fig.tight_layout()
    return fig_to_pil(fig)

def plot_confusion_matrix():
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4), facecolor=BG)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title('혼동 행렬 (Confusion Matrix)', fontsize=12, fontweight='bold')
    fig.set_facecolor(BG)
    fig.tight_layout()
    return fig_to_pil(fig)

In [11]:
feat_ranges = {
    fname: (
        round(X_raw[fname].min(), 1),
        round(X_raw[fname].max(), 1),
        round(X_raw[fname].mean(), 2)
    )
    for fname in selected_features
}

def predict_raw(v1, v2, v3):
    raw = np.array([[v1, v2, v3]])
    sel_idx = np.where(selected_mask)[0]
    means = scaler.mean_[sel_idx]
    stds  = scaler.scale_[sel_idx]
    scaled = (raw - means) / stds

    pred_class = model.predict(scaled)[0]
    pred_proba = model.predict_proba(scaled)[0]

    emoji_map = {'setosa': '🌺', 'versicolor': '🌼', 'virginica': '🌸'}
    name = class_names[pred_class]
    result = f'{emoji_map.get(name, "🌿")} 예측: {name.upper()}\n\n'
    for cname, prob in zip(class_names, pred_proba):
        filled = int(prob * 30)
        bar = '█' * filled + '░' * (30 - filled)
        result += f'{cname:12s} [{bar}] {prob*100:.1f}%\n'

    proba_dict = {cname: float(p) for cname, p in zip(class_names, pred_proba)}
    return result, proba_dict

info_md = f"""
## 📋 모델 정보
| 항목 | 내용 |
|------|------|
| 데이터셋 | Iris (150 샘플) |
| 전처리 | StandardScaler + SelectKBest (k=3) |
| 선택된 피처 | `{selected_features[0]}`, `{selected_features[1]}`, `{selected_features[2]}` |
| 제거된 피처 | `{removed_features[0]}` |
| 모델 | Random Forest (n_estimators=100) |
| 테스트 정확도 | **{accuracy*100:.2f}%** |
"""

with gr.Blocks(title='ML 예측 모델', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🌸 붓꽃(Iris) 분류 예측 모델')
    gr.Markdown('SelectKBest로 선택된 **3개 피처** 입력 → 품종 예측')

    with gr.Tab('🔮 예측하기'):
        gr.Markdown(info_md)
        f0n, f1n, f2n = selected_features
        f0r, f1r, f2r = feat_ranges[f0n], feat_ranges[f1n], feat_ranges[f2n]
        with gr.Row():
            s0 = gr.Slider(minimum=f0r[0], maximum=f0r[1], value=f0r[2], step=0.1, label=f'① {f0n} (cm)')
            s1 = gr.Slider(minimum=f1r[0], maximum=f1r[1], value=f1r[2], step=0.1, label=f'② {f1n} (cm)')
            s2 = gr.Slider(minimum=f2r[0], maximum=f2r[1], value=f2r[2], step=0.1, label=f'③ {f2n} (cm)')
        btn = gr.Button('🚀 예측 실행', variant='primary', size='lg')
        with gr.Row():
            out_text  = gr.Textbox(label='예측 결과', lines=6)
            out_label = gr.Label(label='클래스별 확률', num_top_classes=3)
        btn.click(fn=predict_raw, inputs=[s0, s1, s2], outputs=[out_text, out_label])

    with gr.Tab('📊 Feature Selection'):
        gr.Markdown('### F-score: 값이 클수록 타겟과 연관성 높은 피처')
        score_img = gr.Image(label='F-score 비교', type='pil')
        gr.Button('그래프 보기').click(fn=plot_feature_scores, inputs=[], outputs=[score_img])

    with gr.Tab('📈 모델 평가'):
        with gr.Row():
            with gr.Column():
                gr.Markdown('### 피처 중요도')
                imp_img = gr.Image(label='Feature Importance', type='pil')
                gr.Button('피처 중요도 보기').click(fn=plot_feature_importance, inputs=[], outputs=[imp_img])
            with gr.Column():
                gr.Markdown('### 혼동 행렬')
                cm_img = gr.Image(label='Confusion Matrix', type='pil')
                gr.Button('혼동 행렬 보기').click(fn=plot_confusion_matrix, inputs=[], outputs=[cm_img])
        gr.Markdown(f'### 분류 리포트\n```\n{report}\n```')

demo.launch(share=True)  # ← 공개 URL 생성

/tmp/ipykernel_1426/923963588.py:43: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title='ML 예측 모델', theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bcc34d6a9eddbb64f7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
